# Tennis Match Prediction Chatbot

An independent Python machine-learning project that combines tennis data collection,
feature engineering, logistic regression, model evaluation, and an interactive
command-line chatbot.

> **Data and evaluation:** the model trains on real ATP match statistics from Jeff Sackmann's
> [tennis_atp](https://github.com/JeffSackmann/tennis_atp) dataset (CC BY-NC-SA 4.0). It never uses simulated data.
> Features are built only from each player's *earlier* matches, and the model is tested on *later* matches
> than it was trained on, then compared against simple baselines.


## 1. Setup

The project uses Pandas and NumPy for data processing, Requests and BeautifulSoup
for data collection, and scikit-learn for model training and evaluation.


In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np
import requests
from bs4 import BeautifulSoup
import time
import io
import random
import re


## 2. Player-name normalization

Normalizes player names so scraped rankings and historical records can be matched more consistently.


In [ ]:
# --- Helper Function for Player Name Normalization ---
def _normalize_player_name(name: str) -> str:
    """Normalizes player names for consistent matching (lowercase, strip, basic cleanup)."""
    name = name.lower().strip()
    # Remove any parenthetical additions (like " (ATP)")
    name = re.sub(r'\s+\(.*?\)$', '', name)
    # Remove leading/trailing periods, often seen with initials in some datasets
    name = name.strip('.')
    # Consider replacing common special characters or multiple spaces with single spaces
    name = re.sub(r'\s+', ' ', name)
    return name


## 3. ATP rankings scraper

Retrieves ATP singles rankings, caches results, and uses a fallback player list if scraping fails.


In [ ]:
# --- 0. Web Scraping for Live Player Rankings ---
class ATPScraper:
    def __init__(self):
        self.rankings_url = "https://www.atptour.com/en/rankings/singles"
        self.player_rankings = [] # Stores normalized names
        self.last_scrape_time = 0 # To prevent overly frequent scraping

    def _fetch_html(self, url: str) -> str:
        """Fetches URL content with delay and error handling."""
        try:
            time.sleep(1) # Be polite
            response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0'}, timeout=10)
            response.raise_for_status()
            return response.text
        except requests.exceptions.RequestException as e:
            print(f"Error fetching {url}: {e}")
            return ""

    def scrape_player_rankings(self, num_players: int = 100, force_scrape: bool = False) -> list:
        """Scrapes top N player names and ranks. Caches results."""
        # Use cached rankings if available and not forced to scrape
        if not force_scrape and self.player_rankings and (time.time() - self.last_scrape_time < 3600):
            return self.player_rankings

        print(f"Scraping top {num_players} player rankings from ATP Tour...")
        html = self._fetch_html(self.rankings_url)

        scraped_players = []
        if html:
            soup = BeautifulSoup(html, 'html.parser')
            rows = soup.select('table.ranking-table tbody tr')
            for row in rows:
                if len(scraped_players) >= num_players: break
                try:
                    rank = int(row.select_one('td.rank-cell').get_text(strip=True))
                    name_element = row.select_one('td.player-cell a')
                    original_name = name_element.get_text(strip=True) if name_element else row.select_one('td.player-cell').get_text(strip=True)
                    normalized_name = _normalize_player_name(original_name)
                    scraped_players.append({"name": normalized_name, "original_name": original_name, "rank": rank})
                except (ValueError, AttributeError):
                    # Skip rows that don't conform to expected structure
                    continue

        if not scraped_players:
            print("Failed to scrape any rankings. Using fallback players.")
            fallback_players_raw = [
                "Novak Djokovic", "Carlos Alcaraz", "Jannik Sinner", "Daniil Medvedev",
                "Alexander Zverev", "Andrey Rublev", "Casper Ruud", "Hubert Hurkacz",
                "Taylor Fritz", "Alex de Minaur", "Grigor Dimitrov", "Stefanos Tsitsipas",
                "Tommy Paul", "Ben Shelton", "Frances Tiafoe", "Holger Rune",
                "Felix Auger-Aliassime", "Karen Khachanov", "Lorenzo Musetti", "Cameron Norrie",
                "Denis Shapovalov", "Diego Schwartzman", "Gael Monfils", "Andy Murray",
                "Stan Wawrinka", "Richard Gasquet", "Fabio Fognini", "Marin Cilic",
                "John Isner", "Kevin Anderson", "Juan Martin del Potro", "Nick Kyrgios"
            ]
            self.player_rankings = [{"name": _normalize_player_name(p), "original_name": p, "rank": i+1} for i, p in enumerate(fallback_players_raw)]
        else:
            self.player_rankings = scraped_players

        print(f"Scraped {len(self.player_rankings)} player rankings (live or fallback).")
        self.last_scrape_time = time.time()
        return self.player_rankings

    def get_player_names(self) -> list:
        # This function should return user-friendly names, not necessarily the normalized ones.
        if not self.player_rankings:
            self.scrape_player_rankings()
        return sorted(list(set([p['original_name'] for p in self.player_rankings]))) # Return original names for display

    def get_player_rank(self, player_name: str) -> int | None:
        if not self.player_rankings or (time.time() - self.last_scrape_time > 3600):
            self.scrape_player_rankings()

        normalized_player_name = _normalize_player_name(player_name)

        # Try exact match first
        for p in self.player_rankings:
            if p['name'] == normalized_player_name:
                return p['rank']

        # If no exact match, try partial matching (e.g., "djokovic" matches "novak djokovic")
        for p in self.player_rankings:
            if normalized_player_name in p['name'] or p['name'] in normalized_player_name:
                return p['rank']

        return None # No rank found


## 4. Historical match data and player statistics

Downloads real ATP match files, cleans them, and computes each player's pre-match form (average serve and return statistics over their earlier matches). Raises an error instead of falling back to simulated data.


In [ ]:
# --- 1. Data Acquisition and Feature Preparation ---
class TennisDataLoader:
    """Loads real ATP match statistics and builds leakage-free, pre-match features.

    Data: Jeff Sackmann's tennis_atp dataset (CC BY-NC-SA 4.0)
    https://github.com/JeffSackmann/tennis_atp
    No simulated data is used: if the real data cannot be loaded, this raises an error.
    """
    SEASONS = list(range(2019, 2025))
    URL_TEMPLATE = "https://raw.githubusercontent.com/JeffSackmann/tennis_atp/master/atp_matches_{year}.csv"
    REQUIRED_COLS = [
        'tourney_date', 'match_num', 'round', 'winner_name', 'loser_name', 'winner_rank', 'loser_rank',
        'w_ace', 'w_df', 'w_svpt', 'w_1stIn', 'w_1stWon', 'w_2ndWon',
        'l_ace', 'l_df', 'l_svpt', 'l_1stIn', 'l_1stWon', 'l_2ndWon',
    ]
    ROUND_ORDER = {'R128': 1, 'R64': 2, 'R32': 3, 'R16': 4, 'RR': 4, 'QF': 5, 'SF': 6, 'BR': 7, 'F': 7}
    STAT_COLS = [
        'first_serve_percentage', 'first_serve_points_won_percentage',
        'second_serve_points_won_percentage', 'return_points_won_percentage',
        'ace_rate', 'double_fault_rate',
    ]
    MIN_PRIOR_MATCHES = 5  # both players need this many earlier matches before a match is used

    def __init__(self, atp_scraper):
        self.atp_scraper = atp_scraper
        self.match_data = self._load_historical_match_data()
        self.player_stats_cache = {}
        self._player_rows = self._build_player_match_rows()
        self._training_df = None

    # ---------- loading and cleaning ----------
    def _load_historical_match_data(self) -> pd.DataFrame:
        frames = []
        for year in self.SEASONS:
            url = self.URL_TEMPLATE.format(year=year)
            try:
                response = requests.get(url, timeout=30)
                response.raise_for_status()
                df = pd.read_csv(io.StringIO(response.text), low_memory=False)
                print(f"Loaded {len(df)} matches from {year}.")
                frames.append(df)
            except Exception as e:
                print(f"Could not load {year}: {e}")

        if not frames:
            raise RuntimeError("Could not download any ATP match files. Check your internet connection and that "
                               "the dataset URL still exists. (This version never falls back to simulated data.)")

        df = pd.concat(frames, ignore_index=True)
        missing = [c for c in self.REQUIRED_COLS if c not in df.columns]
        if missing:
            raise RuntimeError(f"The dataset is missing expected columns: {missing}")

        df['winner_name'] = df['winner_name'].astype(str).apply(_normalize_player_name)
        df['loser_name'] = df['loser_name'].astype(str).apply(_normalize_player_name)

        serve_cols = [f'{s}_{c}' for s in ('w', 'l') for c in ('ace', 'df', 'svpt', '1stIn', '1stWon', '2ndWon')]
        before = len(df)
        df = df.dropna(subset=serve_cols)
        df = df[(df['w_svpt'] > df['w_1stIn']) & (df['w_1stIn'] > 0) &
                (df['l_svpt'] > df['l_1stIn']) & (df['l_1stIn'] > 0)]
        df['round_order'] = df['round'].map(self.ROUND_ORDER).fillna(0)
        df = df.sort_values(['tourney_date', 'round_order', 'match_num']).reset_index(drop=True)
        df['match_id'] = df.index
        print(f"Using {len(df)} of {before} matches that have complete serve statistics.")
        return df

    # ---------- per-player, per-match rows ----------
    def _build_player_match_rows(self) -> pd.DataFrame:
        d = self.match_data
        parts = []
        for side, opp, name_col, opp_col, rank_col, won in (
                ('w', 'l', 'winner_name', 'loser_name', 'winner_rank', 1),
                ('l', 'w', 'loser_name', 'winner_name', 'loser_rank', 0)):
            svpt, first_in = d[f'{side}_svpt'], d[f'{side}_1stIn']
            opp_svpt = d[f'{opp}_svpt']
            parts.append(pd.DataFrame({
                'match_id': d['match_id'],
                'tourney_date': d['tourney_date'],
                'player': d[name_col],
                'opponent': d[opp_col],
                'rank': d[rank_col],
                'won': won,
                'first_serve_percentage': first_in / svpt,
                'first_serve_points_won_percentage': d[f'{side}_1stWon'] / first_in,
                'second_serve_points_won_percentage': d[f'{side}_2ndWon'] / (svpt - first_in),
                'return_points_won_percentage': (opp_svpt - d[f'{opp}_1stWon'] - d[f'{opp}_2ndWon']) / opp_svpt,
                'ace_rate': d[f'{side}_ace'] / svpt,
                'double_fault_rate': d[f'{side}_df'] / svpt,
            }))
        long = pd.concat(parts, ignore_index=True).sort_values(['match_id', 'won'], ascending=[True, False])
        long = long.reset_index(drop=True)

        # Pre-match form = average over the player's EARLIER matches only (current match excluded)
        grouped = long.groupby('player')
        long['prior_matches'] = grouped.cumcount()
        for s in self.STAT_COLS:
            prior_sum = grouped[s].cumsum() - long[s]
            long[f'prior_{s}'] = prior_sum / long['prior_matches'].replace(0, np.nan)
        return long

    # ---------- chatbot lookups ----------
    def _resolve_player(self, player_name_raw: str):
        name = _normalize_player_name(player_name_raw)
        names = self._player_rows['player'].unique()
        if name in set(names):
            return name
        matches = [n for n in names if name and name in n]  # e.g. "djokovic" -> "novak djokovic"
        return matches[0] if len(matches) == 1 else None

    def get_player_stats(self, player_name_raw: str) -> dict:
        """Career averages over the loaded seasons (the same quantity the model sees as 'form')."""
        player = self._resolve_player(player_name_raw)
        if player is None:
            return {}
        if player in self.player_stats_cache:
            return self.player_stats_cache[player]
        rows = self._player_rows[self._player_rows['player'] == player]
        stats = {c: rows[c].mean() for c in self.STAT_COLS}
        wins = int(rows['won'].sum())
        stats['win_loss_record'] = f"{wins}-{len(rows) - wins}"
        stats['matches_played'] = len(rows)
        ranked = rows['rank'].dropna()
        stats['dataset_rank'] = float(ranked.iloc[-1]) if len(ranked) else None  # latest rank in the data
        stats['current_rank'] = self.atp_scraper.get_player_rank(player_name_raw)  # live rank, display only
        self.player_stats_cache[player] = stats
        return stats

    def get_all_player_names(self) -> list:
        return sorted(n.title() for n in self._player_rows['player'].unique())

    # ---------- training table ----------
    def get_match_data_for_training(self) -> pd.DataFrame:
        """One row per match: player1 - player2 differences in PRE-MATCH averages (no information from the match itself)."""
        if self._training_df is not None:
            return self._training_df.copy()
        rows = self._player_rows
        win = rows[rows['won'] == 1].set_index('match_id')
        lose = rows[rows['won'] == 0].set_index('match_id')
        m = win.join(lose, lsuffix='_w', rsuffix='_l', how='inner')

        rng = np.random.RandomState(42)
        flip = rng.rand(len(m)) < 0.5  # randomly choose which player is "player 1" so labels are balanced
        out = pd.DataFrame({'tourney_date': m['tourney_date_w'].values}, index=m.index)
        out['player1_name'] = np.where(flip, m['player_l'], m['player_w'])
        out['player2_name'] = np.where(flip, m['player_w'], m['player_l'])
        out['player1_wins'] = np.where(flip, 0, 1)
        sign = np.where(flip, -1.0, 1.0)  # +1 when player 1 is the match winner
        for s in self.STAT_COLS:
            out[f'{s}_diff'] = sign * (m[f'prior_{s}_w'] - m[f'prior_{s}_l'])
        rank1 = np.where(flip, m['rank_l'], m['rank_w']).astype(float)
        rank2 = np.where(flip, m['rank_w'], m['rank_l']).astype(float)
        out['rank1'], out['rank2'] = rank1, rank2
        out['log_rank_diff'] = np.log(rank2) - np.log(rank1)  # positive when player 1 is ranked higher
        out['min_prior'] = np.minimum(m['prior_matches_w'], m['prior_matches_l'])

        out = out[out['min_prior'] >= self.MIN_PRIOR_MATCHES].dropna()
        out = out.drop(columns='min_prior').sort_values('tourney_date').reset_index(drop=True)
        self._training_df = out
        print(f"Training table: {len(out)} matches (each uses only information available before the match).")
        return out.copy()


## 5. Match prediction model

Trains a logistic regression classifier on player-to-player differences in pre-match form and ranking, evaluates it on a later, held-out time period, and compares it against baselines.


In [ ]:
# --- 2. Match Prediction Model ---
class MatchPredictor:
    FEATURES = [
        "first_serve_percentage_diff",
        "first_serve_points_won_percentage_diff",
        "second_serve_points_won_percentage_diff",
        "return_points_won_percentage_diff",
        "ace_rate_diff",
        "double_fault_rate_diff",
        "log_rank_diff",
    ]

    def __init__(self):
        self.model = None
        self.features = self.FEATURES
        self.report = {}

    def train_model(self, df: pd.DataFrame):
        print("Training prediction model...")
        if df.empty or len(df) < 50:
            print("Not enough matches to train and test the model.")
            self.model = None
            return

        # Chronological split: train on earlier matches, test on later ones (never the reverse).
        df = df.sort_values('tourney_date').reset_index(drop=True)
        split_date = df['tourney_date'].iloc[int(len(df) * 0.8)]
        train, test = df[df['tourney_date'] < split_date], df[df['tourney_date'] >= split_date]
        if train.empty or test.empty or train['player1_wins'].nunique() < 2:
            print("Could not form a valid train/test split.")
            self.model = None
            return

        self.model = Pipeline([('scale', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])
        self.model.fit(train[self.features], train['player1_wins'])

        y_test = test['player1_wins']
        accuracy = accuracy_score(y_test, self.model.predict(test[self.features]))
        rank_baseline = accuracy_score(y_test, (test['rank1'] < test['rank2']).astype(int))
        majority = max(y_test.mean(), 1 - y_test.mean())
        self.report = {
            'n_train': len(train), 'n_test': len(test), 'accuracy': accuracy,
            'rank_baseline': rank_baseline, 'majority_baseline': majority,
            'train_end': int(train['tourney_date'].max()), 'test_start': int(split_date),
        }
        print(self.report_text())

    def report_text(self) -> str:
        r = self.report
        if not r:
            return "No evaluation yet. Run 'train model' first."
        return (f"Evaluation on {r['n_test']} held-out later matches (trained on {r['n_train']} earlier matches):\n"
                f"- Model accuracy: {r['accuracy']:.1%}\n"
                f"- Baseline, always pick the higher-ranked player: {r['rank_baseline']:.1%}\n"
                f"- Baseline, always guess one side: {r['majority_baseline']:.1%}")

    def predict_match_outcome(self, p1_stats: dict, p2_stats: dict, p1_name: str, p2_name: str) -> str:
        if self.model is None: return "Prediction model not trained. Type 'train model'."
        if not p1_stats or not p2_stats: return "Cannot predict: Missing stats for one or both players."
        r1, r2 = p1_stats.get('dataset_rank'), p2_stats.get('dataset_rank')
        if r1 is None or r2 is None: return "Cannot predict: no ranking data for one or both players."

        features_input = {f"{s}_diff": p1_stats[s] - p2_stats[s] for s in TennisDataLoader.STAT_COLS}
        features_input['log_rank_diff'] = float(np.log(r2) - np.log(r1))
        pred_df = pd.DataFrame([features_input])[self.features]

        p1_prob = float(self.model.predict_proba(pred_df)[0][1])
        p2_prob = 1 - p1_prob
        winner, w_prob, loser = (p1_name, p1_prob, p2_name) if p1_prob > p2_prob else (p2_name, p2_prob, p1_name)
        return (f"Predicted Winner: **{winner}** (Probability: {w_prob:.1%})\n"
                f"Against {loser} (Probability: {1 - w_prob:.1%})")


## 6. Interactive chatbot

Provides commands for viewing player stats, listing players, training the model, and requesting matchup predictions.


In [ ]:
# --- 3. Chatbot Interface ---
class TennisChatbot:
    def __init__(self):
        self.atp_scraper = ATPScraper()
        # Ensure rankings are scraped immediately upon init for initial use
        self.atp_scraper.scrape_player_rankings(num_players=100, force_scrape=True)
        self.data_loader = TennisDataLoader(self.atp_scraper)
        self.predictor = MatchPredictor()
        self.trained = False

    def process_command(self, command: str) -> str:
        cmd = command.lower().strip()
        if re.match(r"^(hello|hi|hey)\b", cmd): return "Hello my name is Bill! I provide tennis stats and predictions. How can I help?"
        elif "stats for" in cmd:
            player_name_raw = cmd.replace("stats for", "").strip()
            return self._get_player_stats_response(player_name_raw)
        elif "list players" in cmd or "who are the players" in cmd: return self._list_players_response()
        elif "train model" in cmd: return self._train_model_response()
        elif "predict match" in cmd: return "To predict, use 'predict [Player1] vs [Player2]'."
        elif "predict" in cmd and "vs" in cmd: return self._handle_prediction_request(cmd)
        elif "help" in cmd: return self._help_message()
        elif "exit" in cmd or "bye" in cmd: return "Goodbye! See you on the court."
        return "I'm not sure how to handle that. Try 'help'."

    def _get_player_stats_response(self, player_name_raw: str) -> str:
        stats = self.data_loader.get_player_stats(player_name_raw)

        display_player_name = player_name_raw.title()

        if not stats:
            return f"Sorry, no historical stats found for **{display_player_name}**. This could be because there's no data for them in the historical dataset, or their name might be spelled differently. Try 'list players' for available names."

        response = f"Stats for **{display_player_name}** (historical data, ATP tour-level matches loaded):\n"

        rank_info = stats.get("current_rank")
        if rank_info is not None:
            response += f"- **Current Rank**: {rank_info}\n"
        else:
            response += f"- **Current Rank**: Not available (Could not retrieve live ranking for this player. Try slightly different spelling or a player from 'list players').\n"

        stat_keys_to_display = [
            'first_serve_percentage', 'first_serve_points_won_percentage',
            'second_serve_points_won_percentage', 'return_points_won_percentage',
            'ace_rate', 'double_fault_rate', 'win_loss_record', 'matches_played'
        ]
        labels = {
            'first_serve_percentage': 'First Serve In', 'first_serve_points_won_percentage': 'First Serve Points Won',
            'second_serve_points_won_percentage': 'Second Serve Points Won', 'return_points_won_percentage': 'Return Points Won',
            'ace_rate': 'Aces (per serve point)', 'double_fault_rate': 'Double Faults (per serve point)',
            'win_loss_record': 'Win-Loss Record', 'matches_played': 'Matches Played',
        }
        for k in stat_keys_to_display:
            value = stats.get(k)
            if value is None or (isinstance(value, float) and pd.isna(value)):
                response += f"- {labels[k]}: Data not available\n"
            elif isinstance(value, float):
                response += f"- {labels[k]}: {value:.1%}\n"
            else:
                response += f"- {labels[k]}: {value}\n"
        return response

    def _list_players_response(self) -> str:
        live_players_info = [{"name": p['original_name'], "rank": p['rank']} for p in self.atp_scraper.player_rankings[:20]]

        # Get all historical names, converting them back to a presentable Title Case
        historical_players_set_normalized = set(self.data_loader.match_data['winner_name'].tolist() + self.data_loader.match_data['loser_name'].tolist()) if not self.data_loader.match_data.empty else set()

        all_display_names = set()
        # Add live player original names (which are already display-friendly)
        for p_info in self.atp_scraper.player_rankings:
            all_display_names.add(p_info['original_name'])

        # Add historical player names. Prioritize matching with scraped original names
        # for consistency, otherwise title case the normalized name.
        for normalized_h_name in historical_players_set_normalized:
            found_match = False
            for p_info in self.atp_scraper.player_rankings:
                if normalized_h_name == p_info['name']: # Check if normalized historical matches normalized live
                    all_display_names.add(p_info['original_name']) # Add the original live name
                    found_match = True
                    break
            if not found_match:
                # If no match with live names, just title case the normalized historical name
                all_display_names.add(normalized_h_name.title())

        sorted_all_display_names = sorted(list(all_display_names))

        response_parts = []
        if live_players_info:
            response_parts.append("Players (top 20 live rankings first):\n" + "".join([f"- **{p['name']}** (Rank: {p['rank']})\n" for p in live_players_info]))

        # Filter historical players to only include those not already in the top 20 live list
        live_player_names_normalized = {_normalize_player_name(p['name']) for p in live_players_info}
        other_historical_players_display_filtered = sorted([p for p in sorted_all_display_names if _normalize_player_name(p) not in live_player_names_normalized])

        if other_historical_players_display_filtered:
            response_parts.append("\nAnd historical data for others including: " + ", ".join(other_historical_players_display_filtered[:10]) + ("..." if len(other_historical_players_display_filtered) > 10 else ""))

        return "".join(response_parts) if response_parts else "No player data available. This might be due to an issue loading historical data or scraping live rankings."


    def _train_model_response(self) -> str:
        if self.trained: return "Model already trained."
        if self.data_loader.match_data.empty: return "Cannot train: No historical data loaded. Check the data source."
        try:
            training_data = self.data_loader.get_match_data_for_training()
            if training_data.empty:
                return "Cannot train: Prepared training data is empty after cleaning. This could mean critical columns are missing or all rows were dropped."
            self.predictor.train_model(training_data)
            self.trained = True
            return "Prediction model trained successfully!\n" + self.predictor.report_text()
        except Exception as e:
            return f"Error during training: {e}. Ensure enough data is available and features can be calculated."

    def _handle_prediction_request(self, cmd: str) -> str:
        if not self.trained: return "Please train the model first ('train model')."
        try:
            players_str = cmd.replace("predict", "").replace("match", "").strip()
            if "vs" not in players_str:
                return "Invalid prediction format. Use 'predict [Player1] vs [Player2]'."

            p1_name_raw, p2_name_raw = [p.strip() for p in players_str.split("vs", 1)]

            p1_stats, p2_stats = self.data_loader.get_player_stats(p1_name_raw), self.data_loader.get_player_stats(p2_name_raw)

            missing_players = []
            if not p1_stats:
                missing_players.append(p1_name_raw.title())
            if not p2_stats:
                missing_players.append(p2_name_raw.title())

            if missing_players:
                return f"Missing historical stats for: {', '.join(missing_players)}. Try 'list players' to see available players."

            return self.predictor.predict_match_outcome(p1_stats, p2_stats, p1_name_raw.title(), p2_name_raw.title())
        except Exception as e:
            return f"Error predicting: {e}. Please ensure player names are correct and the format is 'predict [Player1] vs [Player2]'."

    def _help_message(self) -> str:
        return ("Commands:\n- '**Hello**' or '**Hi**': Greet the bot.\n- '**Stats for [Player Name]**': Get historical statistics for a player.\n- '**List players**' or '**Who are the players**': See a list of top-ranked players and others with historical data.\n"
                "- '**Train model**': Train the match prediction model using historical data.\n- '**Predict [Player1] vs [Player2]**': Predict the winner of a match between two players (model must be trained).\n- '**Help**': Display this help message.\n- '**Exit**' or '**Bye**': End the conversation.")


## 7. Run the chatbot

Starts the command-line interaction loop. Run this cell after the earlier cells.


In [ ]:
# --- Main Interaction Loop ---
def run_chatbot():
    print("Initializing Tennis Chatbot...")
    chatbot = TennisChatbot()
    print("\nBot: " + chatbot.process_command("hello"))
    print("\nBot: " + chatbot.process_command("help"))
    while True:
        user_input = input("\nYou: ")
        response = chatbot.process_command(user_input)
        print(f"Bot: {response}")
        if "goodbye" in response.lower(): break

if __name__ == "__main__":
    run_chatbot()
